# 5.9 阶段七：负载均衡（BN1S1）

最后一个阶段解决分核的遗留问题：**任务数不等于核数时怎么办？**

## 1. BN1 的不均衡

阶段二按 B×N 分核，两种典型失衡：

```text
任务不足：B=1, N=8, 40 核 → blockDim=8，32 核闲置
任务超载：B=8, N=40, 20 核 → 每核领 16 个 (b,n)，尚可均分
```

更麻烦的是**任务不均**：每个 (b, n) 任务的计算量 = QS × KVS，batch 变长（TND、actualSeqLengths）时各任务量不同——均分任务数 ≠ 均分工作量。

## 2. BN1S1：把 S1 也切成任务

解决思路：把 Q 的 S1 轴也加入切分维度，任务粒度变细：

```text
totalSize = B × N × ceil(QS / s1Block)     ← 多了 S1 维
splitFactorSize = ceil(totalSize / 实际用核数)
核 i 领任务区间 [i × splitFactorSize, (i+1) × splitFactorSize)
```

对照第 3 章 3.5 节 `SetMultiCoreParams`：`totalSize = bSize × n2Size × gSize × s1OuterSize`——正是 BN1S1 的通用版（GQA 下用 N2×G 替代 N）。

两个工程细节：

1. **向上取整切分**：`ceil(totalSize / actualAiv)` 保证「最差的基本块性能最优」（第 3 章 3.5 节）；
2. **尾差至多 1 块**：totalSize 不整除时前面的核多领一块，最坏不均衡被控制在单块级别。

## 3. 更远的路（第 9 章预告）

BN1S1 仍非终点。当 batch 变长 + GQA + 尾块叠加时，任务量差异进一步拉大，需要更精细的负载均衡：

- **带 section 的切分**：按计算量而非任务数分组；
- **GS1 合轴 / 不合轴**：GQA 的组维度与 S1 是否合并考虑；
- **切 S2 负载均衡**：终极手段，S2 也切给多核 + 归并（FlashDecode 思路的工程化）。

这些是第 9 章「性能优化高阶」的内容。

## 4. 七阶段回顾

```text
一 基础极简 FA   ：跑对（单核串行，四拍循环）
二 BN1 简单分核  ：跑多（B×N 任务分核）
三 buffer 复用   ：搬运计算重叠（Q 常驻 + KV 乒乓）
四 preload 流水  ：Cube/Vector 交叉掩盖（无断流）
五 mask          ：功能性补全（sparsemode3 块级跳过）
六 尾块          ：变长支持（QS/KVS 非 128 倍数）
七 负载均衡      ：任务均衡（BN1S1 细粒度分核）
```

至此，一个非量化 FA 算子的完整能力面就绪。第 6 章进入量化世界。

## 小测验

1. BN1S1 相比 BN1 多切了哪个轴？totalSize 公式多了哪一项？
2. `splitFactorSize` 向上取整 vs 向下取整，对「最差的核」分别意味着什么？
3. 任务数不均与任务量不均分别靠什么解决？